In [1]:
import itertools

import pandas as pd
import pm4py

## Attribute functional relationship

Problem: Identify functional relationships between attributes and characterize the cardinality of the relationships between their values.

Motivation: Event-log attributes may encode related information at different levels of granularity. For example, one attribute may uniquely determine another, while several values of one attribute may correspond to the same value of another. Identifying such relationships helps analysts understand dependencies between attributes, recognize equivalent or hierarchically related information, and avoid treating dependent attributes as independent dimensions in subsequent analyses.

Approach: For each pair of attributes A and B, examine the mappings between their observed values. Determine whether each value of A uniquely determines a value of B and whether each value of B uniquely determines a value of A. Based on these properties, characterize the observed relationship as one-to-one, one-to-many, many-to-one, or many-to-many.

Output: Attribute pairs exhibiting functional relationships, together with their relationship cardinality, support, and the corresponding mappings between observed values. A separate diagnostic table lists every remaining pair with the reason it is not functional (trivial, disjoint, or many-to-many).

Requirements:
- for each pair of attributes, only rows where both attributes are populated are considered
- the relationship is determined over the distinct value pairs observed on those rows
- A functionally determines B if each value of A co-occurs with exactly one value of B; B functionally determines A symmetrically
- pairs are characterized as one-to-one (functional in both directions), many-to-one or one-to-many (functional in one direction), or many-to-many (neither)
- pairs where either attribute has fewer than two distinct jointly-populated values are characterized as trivial (e.g. constant attributes); pairs that are never jointly populated are characterized as disjoint
- support is the number of rows where both attributes are populated divided by the total number of rows in the log (the common association-rule definition), reported for every pair as the share of the log backing the observed relationship
- every attribute pair is classified; only pairs that are functional in at least one direction are reported as functional relationships, the rest are listed separately with their reason

In [2]:
event_log = pm4py.read_xes('../../data/BPI_Challenge_2017.xes')
# Road_Traffic_Fine_Management_Process.xes, DomesticDeclarations, SepsisCases2020EventLog.xes, BPIC2011_hospital_log.xes, BPI_Challenge_2012.xes, BPIC15_1.xes, BPI_Challenge_2017.xes, BPI_Challenge_2018.xes
event_log = event_log.drop(columns=['lifecycle:transition'], errors='ignore')  # drop distracting columns, ignore errors if nonexisting

CASE_ID = 'case:concept:name'
ACTIVITY = 'concept:name'
COMPLETION_TIME = 'time:timestamp'

display(event_log)

C:\Users\MFranceschetti\PycharmProjects\epm-patterns\.venv\Lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

,Action,org:resource,concept:name,EventOrigin,EventID,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,2016-01-01 09:51:15.304000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,2016-01-01 09:51:15.352000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,2016-01-01 09:51:15.774000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1673366067,2016-01-01 09:52:36.392000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,2016-01-01 09:52:36.403000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1202262,Deleted,User_1,W_Call after offers,Workflow,Workitem_1817549786,2017-01-06 06:33:02.212000+00:00,Home improvement,New credit,Application_1350494635,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1202263,Created,User_1,W_Call after offers,Workflow,Workitem_363876066,2017-01-06 06:33:02.221000+00:00,Home improvement,New credit,Application_1350494635,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1202264,statechange,User_28,A_Cancelled,Application,ApplState_1869071797,2017-01-16 09:51:21.114000+00:00,Home improvement,New credit,Application_1350494635,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1202265,statechange,User_28,O_Cancelled,Offer,OfferState_420066181,2017-01-16 09:51:21.139000+00:00,Home improvement,New credit,Application_1350494635,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Offer_1580299144


In [ ]:
def functional_relationship(series_a, series_b):
    """Characterize the value-level relationship between two attributes over jointly-populated rows.

    Always returns a dict describing the pair. 'cardinality' is one of:
    - 'one-to-one'    : functional in both directions
    - 'many-to-one'   : a functionally determines b only
    - 'one-to-many'   : b functionally determines a only
    - 'many-to-many'  : neither direction is functional (not a functional relationship)
    - 'trivial'       : fewer than two distinct values on either side
    - 'disjoint'      : the two attributes are never jointly populated

    'direction' and 'mapping' are populated only for the three functional cardinalities and are None
    otherwise. 'n_a', 'n_b', 'n_pairs' are the distinct-value and distinct-pair counts over the
    jointly-populated rows. 'n_rows' is the number of rows where both attributes are populated and
    'support' is that count divided by the total number of rows (the common association-rule
    definition), i.e. the share of the log that backs the observed relationship. The non-functional
    cardinalities are reported for transparency and filtered out at display time.
    """
    mask = series_a.notna() & series_b.notna()
    n_rows = int(mask.sum())
    support = n_rows / len(series_a) if len(series_a) else 0.0
    pairs = pd.DataFrame({'a': series_a[mask], 'b': series_b[mask]}).drop_duplicates()

    n_a, n_b, n_pairs = pairs['a'].nunique(), pairs['b'].nunique(), len(pairs)
    result = {'cardinality': None, 'direction': None, 'mapping': None,
              'n_a': n_a, 'n_b': n_b, 'n_pairs': n_pairs, 'n_rows': n_rows, 'support': support}

    if n_pairs == 0:
        return {**result, 'cardinality': 'disjoint'}

    if n_a < 2 or n_b < 2:
        return {**result, 'cardinality': 'trivial'}  # fewer than two distinct values on either side

    a_determines_b = pairs.groupby('a')['b'].nunique().max() == 1
    b_determines_a = pairs.groupby('b')['a'].nunique().max() == 1

    if a_determines_b and b_determines_a:
        cardinality, direction, mapping = 'one-to-one', 'a_to_b', pairs.set_index('a')['b'].to_dict()
    elif a_determines_b:
        cardinality, direction, mapping = 'many-to-one', 'a_to_b', pairs.set_index('a')['b'].to_dict()
    elif b_determines_a:
        cardinality, direction, mapping = 'one-to-many', 'b_to_a', pairs.set_index('b')['a'].to_dict()
    else:
        return {**result, 'cardinality': 'many-to-many'}  # not a functional relationship

    return {**result, 'cardinality': cardinality, 'direction': direction, 'mapping': mapping}

In [ ]:
columns = event_log.columns.tolist()

FUNCTIONAL_CARDINALITIES = ['one-to-one', 'many-to-one', 'one-to-many']

all_pairs = []
for attribute_a, attribute_b in itertools.combinations(columns, 2):
    relationship = functional_relationship(event_log[attribute_a], event_log[attribute_b])
    all_pairs.append({
        'attribute_a': attribute_a,
        'attribute_b': attribute_b,
        'cardinality': relationship['cardinality'],
        'direction': relationship['direction'],
        'n_a': relationship['n_a'],
        'n_b': relationship['n_b'],
        'n_pairs': relationship['n_pairs'],
        'n_rows': relationship['n_rows'],
        'support': relationship['support'],
        'mapping': relationship['mapping'],
    })

all_results = pd.DataFrame(all_pairs)  # every column pair, functional or not

results = all_results[all_results['cardinality'].isin(FUNCTIONAL_CARDINALITIES)].reset_index(drop=True)
display(results.drop(columns='mapping') if not results.empty else results)

In [ ]:
# Diagnostic: every pair that was analyzed but is not a functional relationship, with the reason.
# 'trivial'      -> one of the attributes has < 2 distinct jointly-populated values
# 'disjoint'     -> the two attributes are never populated on the same row
# 'many-to-many' -> both attributes take multiple values for some value of the other
excluded = all_results[~all_results['cardinality'].isin(FUNCTIONAL_CARDINALITIES)].reset_index(drop=True)

print(f"{len(all_results)} pairs analyzed: {len(results)} functional, {len(excluded)} not")
print(excluded['cardinality'].value_counts().to_string())
display(excluded.drop(columns=['mapping', 'direction']))

In [ ]:
from ipywidgets import interact

@interact(pair=list(results.index))
def inspect_mapping(pair):
    row = results.loc[pair]
    if row['direction'] == 'a_to_b':
        src, dst = row['attribute_a'], row['attribute_b']
    else:
        src, dst = row['attribute_b'], row['attribute_a']
    print(f"{row['attribute_a']} <-> {row['attribute_b']}: {row['cardinality']} "
          f"({row['n_a']} distinct {row['attribute_a']} values, {row['n_b']} distinct {row['attribute_b']} values, "
          f"support {row['support']:.4f} over {row['n_rows']} rows)")
    print(f"functional mapping {src} -> {dst}:")
    display(pd.Series(row['mapping'], name=dst).rename_axis(src))